In [1]:
import pandas as pd
import numpy as np


from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from sklearn.impute import SimpleImputer

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    MinMaxScaler,
    StandardScaler,
    RobustScaler,
    OneHotEncoder
)

In [2]:
df = pd.read_parquet('../dados/gercon_poa_pronto.parquet')

In [3]:
df.info(verbose=True, show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 798527 entries, 0 to 798526
Data columns (total 21 columns):
 #   Column                      Non-Null Count   Dtype   
---  ------                      --------------   -----   
 0   corregulador                798415 non-null  category
 1   diasfilaespera              798527 non-null  float64 
 2   especialidademae            798527 non-null  category
 3   cid                         798527 non-null  category
 4   idpaciente                  798527 non-null  str     
 5   idade                       798527 non-null  float64 
 6   cnes_solicitante            798527 non-null  str     
 7   situacaoagenda              798527 non-null  category
 8   cnesexecutante              798527 non-null  str     
 9   possuiunidadeindicada       798527 non-null  category
 10  censo_favela_comunidade     798527 non-null  float64 
 11  censo_taxa_alfabetizacao    798527 non-null  float64 
 12  censo_coordenadoria_saude   798527 non-null  category
 13  censo_dist

In [4]:
df.isna().sum().sort_values(ascending=False)

corregulador                  112
diasfilaespera                  0
especialidademae                0
cid                             0
idpaciente                      0
idade                           0
cnes_solicitante                0
situacaoagenda                  0
cnesexecutante                  0
possuiunidadeindicada           0
censo_favela_comunidade         0
censo_taxa_alfabetizacao        0
censo_coordenadoria_saude       0
censo_distrito_sanitario        0
censo_renda_media_winsor        0
distancia_km                    0
tempo_caminhando                0
distancia_km_trans_publico      0
tempo_desloc_trans_publico      0
quant_bus                       0
baldiacoes                      0
dtype: int64

In [5]:
corregulador_order = {
    'AZUL': 0,
    'AMARELO': 1,
    'LARANJA': 2,
    'VERMELHO': 3
}

df['corregulador'] = df['corregulador'].map(corregulador_order)


In [6]:

imputer = SimpleImputer(strategy='most_frequent')

df[['corregulador']] = imputer.fit_transform(
    df[['corregulador']]
)


'''
df['corregulador'] = (
    df['corregulador']
    .cat.add_categories(['nenhum'])
    .fillna('nenhum')
)
'''

"\ndf['corregulador'] = (\n    df['corregulador']\n    .cat.add_categories(['nenhum'])\n    .fillna('nenhum')\n)\n"

In [7]:
ID_COLUMNS = ['idpaciente']

df = df.drop(columns=ID_COLUMNS)

In [8]:
STR_TO_CAT_COLS = ['cnesexecutante', 'cnes_solicitante']

df[STR_TO_CAT_COLS] = df[STR_TO_CAT_COLS].astype('category')

#### Modelagem

In [9]:
df.info(verbose=True, show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 798527 entries, 0 to 798526
Data columns (total 20 columns):
 #   Column                      Non-Null Count   Dtype   
---  ------                      --------------   -----   
 0   corregulador                798527 non-null  float64 
 1   diasfilaespera              798527 non-null  float64 
 2   especialidademae            798527 non-null  category
 3   cid                         798527 non-null  category
 4   idade                       798527 non-null  float64 
 5   cnes_solicitante            798527 non-null  category
 6   situacaoagenda              798527 non-null  category
 7   cnesexecutante              798527 non-null  category
 8   possuiunidadeindicada       798527 non-null  category
 9   censo_favela_comunidade     798527 non-null  float64 
 10  censo_taxa_alfabetizacao    798527 non-null  float64 
 11  censo_coordenadoria_saude   798527 non-null  category
 12  censo_distrito_sanitario    798527 non-null  category
 13  censo_rend

In [10]:
df['target'] = (df['situacaoagenda'].eq('FALTANTE')).astype('int8')

In [11]:
print(df['target'].value_counts())

target
0    662581
1    135946
Name: count, dtype: int64


In [12]:
X = df.drop(columns=['target'])

In [13]:
y = df['target']

In [14]:
cat_cols = X.select_dtypes(include=['object', 'str', 'category']).columns

num_cols = X.select_dtypes(include=['number']).columns

date_cols = X.select_dtypes(include=['datetime']).columns

print(f"Shape do dataset: {X.shape}")
print(f"Colunas numéricas: {list(num_cols)}")
print(f"Colunas categóricas: {list(cat_cols)}")
print(f"Colunas de data: {list(date_cols)}")

Shape do dataset: (798527, 20)
Colunas numéricas: ['corregulador', 'diasfilaespera', 'idade', 'censo_favela_comunidade', 'censo_taxa_alfabetizacao', 'censo_renda_media_winsor', 'distancia_km', 'tempo_caminhando', 'distancia_km_trans_publico', 'tempo_desloc_trans_publico', 'quant_bus', 'baldiacoes']
Colunas categóricas: ['especialidademae', 'cid', 'cnes_solicitante', 'situacaoagenda', 'cnesexecutante', 'possuiunidadeindicada', 'censo_coordenadoria_saude', 'censo_distrito_sanitario']
Colunas de data: []


In [15]:
cat_cols = X.select_dtypes(
    include=['object', 'str', 'category']
).columns

num_cols = X.select_dtypes(
    include=['number']
).columns

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ]
)

X_processed = preprocessor.fit_transform(X)

In [16]:
X.shape

(798527, 20)

In [17]:
X.head()

,corregulador,diasfilaespera,especialidademae,cid,idade,cnes_solicitante,situacaoagenda,cnesexecutante,possuiunidadeindicada,censo_favela_comunidade,censo_taxa_alfabetizacao,censo_coordenadoria_saude,censo_distrito_sanitario,censo_renda_media_winsor,distancia_km,tempo_caminhando,distancia_km_trans_publico,tempo_desloc_trans_publico,quant_bus,baldiacoes
0,1.0,17.0,CIRURGIA BUCOMAXILOFACIAL,K011,27.0,2265133,REALIZADA,7031092,SIM,10.235504,96.913111,LESTE,LESTE,2760.800651,1.280,18.916667,1.528,12.100000,1.0,0.0
1,0.0,14.0,CIRURGIA BUCOMAXILOFACIAL,K021,53.0,2237911,REALIZADA,7031092,SIM,34.324491,95.207445,LESTE,NORDESTE,1659.903950,5.901,86.800000,6.296,28.183333,1.0,0.0
2,0.0,1.0,CIRURGIA BUCOMAXILOFACIAL,K011,29.0,2264838,REALIZADA,7031092,NÃO,13.014997,97.287075,LESTE,PARTENON,3993.913191,3.083,47.333333,4.328,17.583333,1.0,0.0
3,2.0,5.0,ODONTO PERIODONTIA,K061,46.0,2264404,REALIZADA,7031092,NÃO,14.905920,94.976008,SUL,RESTINGA,1700.161961,14.957,215.316667,18.601,67.266667,2.0,1.0
4,1.0,0.0,ODONTO PERIODONTIA,K051,63.0,6130917,REALIZADA,7031092,NÃO,12.121846,97.716534,OESTE,GLORIA,3517.723782,11.228,164.050000,18.210,54.066667,2.0,1.0


In [18]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)

In [19]:
rf_clf = RandomForestClassifier(n_estimators=501, max_depth=5, random_state=42)

In [20]:
rf_clf.fit(X_train, y_train)

ValueError: could not convert string to float: 'GINECOLOGIA'